# 03 · GOLD — Insights: Correlação IPCA × Boi Gordo

**Entrada:** `etl_pos__silver.economia`  
**Saída:**   `etl_pos__gold.indicadores` (série temporal com variações)  

**Métricas calculadas:**
- `variacao_ipca`  — variação percentual mensal do IPCA (Window · lag)
- `variacao_boi`   — variação percentual mensal do preço do Boi Gordo
- `correlacao`     — Coeficiente de Pearson entre as variações (scalar · collect)

**Hipótese:** Variações no preço do Boi Gordo apresentam correlação positiva com o IPCA,  
dado o peso do sub-índice de carnes na cesta de consumo brasileira.

**ADR-004:** `F.corr()` retorna escalar — exibido via `collect()`, não gravado como tabela.

In [ ]:
# ============================================================
# CELL 1 — Imports e constantes
# ============================================================
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

spark = SparkSession.builder.getOrCreate()

SOURCE  = "etl_pos__silver.economia"
TARGET  = "etl_pos__gold.indicadores"

print(f"Fonte  : {SOURCE}")
print(f"Destino: {TARGET}")

In [ ]:
# ============================================================
# CELL 2 — Leitura da Silver
# ============================================================
insights = spark.table(SOURCE).orderBy("data")

print(f"Registros na Silver: {insights.count()}")
insights.printSchema()
display(insights)

In [ ]:
# ============================================================
# CELL 3 — Window Function · Variação % mês a mês
# lag() pega o valor do mês anterior.
# variacao = (atual - anterior) / anterior * 100
# F.when protege contra divisão por zero e nulls.
# ============================================================

# Janela ordenada por data (sem partição — série única)
w = Window.orderBy("data")

insights_var = (
    insights
    # Valores do mês anterior
    .withColumn("ipca_ant", F.lag("ipca").over(w))
    .withColumn("boi_ant",  F.lag("boi_gordo").over(w))

    # Variação IPCA mês a mês
    .withColumn(
        "variacao_ipca",
        F.when(
            F.col("ipca_ant").isNotNull() & (F.col("ipca_ant") != 0),
            (F.col("ipca") - F.col("ipca_ant")) / F.col("ipca_ant") * 100
        ).otherwise(F.lit(None).cast("double"))
    )

    # Variação Boi Gordo mês a mês
    .withColumn(
        "variacao_boi",
        F.when(
            F.col("boi_ant").isNotNull() & (F.col("boi_ant") != 0),
            (F.col("boi_gordo") - F.col("boi_ant")) / F.col("boi_ant") * 100
        ).otherwise(F.lit(None).cast("double"))
    )

    # Remover colunas intermediárias (lag)
    .drop("ipca_ant", "boi_ant")
    .orderBy("data")
)

print(f"Schema Gold:")
insights_var.printSchema()
print(f"Registros (1o mes null por design): {insights_var.count()}")

In [ ]:
# ============================================================
# CELL 4 — Preview da série de variações
# ============================================================
display(insights_var.select("data", "ipca", "boi_gordo", "variacao_ipca", "variacao_boi"))

In [ ]:
# ============================================================
# CELL 5 — Correlação de Pearson entre variações
# ADR-004: F.corr() retorna escalar → collect()[0] para leitura.
#          Nunca usar toPandas() para calcular correlações.
# ============================================================

# Filtrar apenas registros com ambas variações não-nulas
df_corr = insights_var.filter(
    F.col("variacao_ipca").isNotNull() & F.col("variacao_boi").isNotNull()
)

corr_val = (
    df_corr
    .select(F.corr("variacao_ipca", "variacao_boi").alias("pearson"))
    .collect()[0]["pearson"]
)

# Interpretação qualitativa
if corr_val is None:
    interp = "Nao calculada (dados insuficientes)"
elif abs(corr_val) >= 0.7:
    interp = "FORTE" + (" positiva" if corr_val > 0 else " negativa")
elif abs(corr_val) >= 0.4:
    interp = "MODERADA" + (" positiva" if corr_val > 0 else " negativa")
else:
    interp = "FRACA ou inexistente"

print("=" * 55)
print("CORRELACAO DE PEARSON — variacao_ipca x variacao_boi")
print("=" * 55)
print(f"  Coeficiente   : {corr_val:.4f}" if corr_val else "  Coeficiente : N/A")
print(f"  Interpretacao : {interp}")
print(f"  Registros     : {df_corr.count()} meses")
print("=" * 55)
print()
print("Escala de referencia:")
print("  [0.7, 1.0]  → correlacao forte")
print("  [0.4, 0.7)  → correlacao moderada")
print("  [0.0, 0.4)  → correlacao fraca")
print("  Negativo    → relacao inversa")

In [ ]:
# ============================================================
# CELL 6 — Visualização: Série Temporal dual-axis
# toPandas() APENAS para matplotlib — nunca para cálculos.
# ============================================================

df_plot = (
    insights_var
    .select("data", "ipca", "boi_gordo", "variacao_ipca", "variacao_boi")
    .orderBy("data")
    .toPandas()  # toPandas APENAS para visualização
)

fig, axes = plt.subplots(2, 1, figsize=(14, 10))
fig.patch.set_facecolor("#0d1520")

# --- GRÁFICO 1: Níveis absolutos ---
ax1 = axes[0]
ax1.set_facecolor("#0d1520")

color_ipca = "#22c98a"
color_boi  = "#e8a020"

l1 = ax1.plot(df_plot["data"], df_plot["ipca"],
               color=color_ipca, linewidth=2.5, marker="o", markersize=5, label="IPCA (% a.m.)")
ax1.set_ylabel("IPCA (% a.m.)", color=color_ipca, fontsize=11)
ax1.tick_params(axis="y", labelcolor=color_ipca)
ax1.tick_params(axis="x", colors="#8895a7")
ax1.set_title("IPCA × Boi Gordo — Evolução dos Índices (2024-2025)",
               color="#dde6f0", fontsize=13, pad=14, fontweight="bold")

ax1b = ax1.twinx()
ax1b.plot(df_plot["data"], df_plot["boi_gordo"],
           color=color_boi, linewidth=2.5, marker="s", markersize=5, label="Boi Gordo (R$/arroba)")
ax1b.set_ylabel("Boi Gordo (R$/arroba)", color=color_boi, fontsize=11)
ax1b.tick_params(axis="y", labelcolor=color_boi)

for ax in [ax1, ax1b]:
    ax.spines["bottom"].set_color("#1e2d3d")
    ax.spines["top"].set_color("#1e2d3d")
    ax.spines["left"].set_color("#1e2d3d")
    ax.spines["right"].set_color("#1e2d3d")
    ax.grid(True, alpha=0.15, color="#3a5070")

lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax1b.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2,
            loc="upper right", framealpha=0.3, labelcolor="#dde6f0",
            facecolor="#0d1520", edgecolor="#1e2d3d")

# --- GRÁFICO 2: Variações mensais ---
ax2 = axes[1]
ax2.set_facecolor("#0d1520")

df_var = df_plot.dropna(subset=["variacao_ipca", "variacao_boi"])

x = range(len(df_var))
width = 0.35
bars1 = ax2.bar([i - width/2 for i in x], df_var["variacao_ipca"],
                 width, color=color_ipca, alpha=0.8, label="Var. IPCA (%)")
bars2 = ax2.bar([i + width/2 for i in x], df_var["variacao_boi"],
                 width, color=color_boi, alpha=0.8, label="Var. Boi Gordo (%)")

ax2.set_xticks(list(x))
ax2.set_xticklabels(
    [str(d)[:7] for d in df_var["data"]],
    rotation=45, ha="right", color="#8895a7", fontsize=9
)
ax2.set_ylabel("Variacao % (m/m)", color="#8895a7", fontsize=11)
ax2.tick_params(axis="y", colors="#8895a7")
ax2.axhline(y=0, color="#3a5070", linewidth=1, linestyle="--")
ax2.set_title(
    f"Variacao Mensal — Correlacao de Pearson: {corr_val:.4f} ({interp})",
    color="#dde6f0", fontsize=12, pad=10, fontweight="bold"
)
ax2.legend(framealpha=0.3, labelcolor="#dde6f0",
            facecolor="#0d1520", edgecolor="#1e2d3d")

for spine in ax2.spines.values():
    spine.set_color("#1e2d3d")
ax2.grid(True, alpha=0.15, color="#3a5070")

plt.tight_layout(pad=2.0)
plt.savefig("/tmp/ipca_boi_gordo_insights.png", dpi=150,
             bbox_inches="tight", facecolor=fig.get_facecolor())
plt.show()
print("Grafico salvo em /tmp/ipca_boi_gordo_insights.png")

In [ ]:
# ============================================================
# CELL 7 — Resumo estatístico Gold
# ============================================================
print("=" * 55)
print("RESUMO EXECUTIVO — Gold Layer")
print("=" * 55)
print("\nEstatísticas IPCA:")
insights_var.select(
    F.round(F.mean("ipca"), 4).alias("media"),
    F.round(F.stddev("ipca"), 4).alias("desvio_padrao"),
    F.round(F.min("ipca"), 4).alias("minimo"),
    F.round(F.max("ipca"), 4).alias("maximo"),
).show()

print("Estatísticas Boi Gordo (R$/arroba):")
insights_var.select(
    F.round(F.mean("boi_gordo"), 2).alias("media"),
    F.round(F.stddev("boi_gordo"), 2).alias("desvio_padrao"),
    F.round(F.min("boi_gordo"), 2).alias("minimo"),
    F.round(F.max("boi_gordo"), 2).alias("maximo"),
).show()

print(f"Correlacao de Pearson (variacoes m/m): {corr_val:.4f}")
print(f"Interpretacao                        : {interp}")

In [ ]:
# ============================================================
# CELL 8 — Gravar Gold (Delta · append + overwriteSchema)
# Grava série temporal de variações. Correlação é scalar
# e exibida apenas no notebook (ADR-004).
# ============================================================
(
    insights_var
    .write
    .format("delta")
    .mode("append")
    .option("overwriteSchema", "true")
    .saveAsTable(TARGET)
)

total_gold = spark.table(TARGET).count()
print(f"Tabela '{TARGET}' gravada com sucesso!")
print(f"Total registros Gold: {total_gold}")

In [ ]:
# ============================================================
# CELL 9 — Validação final (critérios TASK-004)
# ============================================================
df_final = spark.table(TARGET).orderBy("data")

print("=" * 50)
print("VALIDACAO TASK-004 — Gold Indicadores")
print("=" * 50)
print(f"  variacao_ipca calculada : {'SIM' if 'variacao_ipca' in df_final.columns else 'NAO'}")
print(f"  variacao_boi calculada  : {'SIM' if 'variacao_boi' in df_final.columns else 'NAO'}")
print(f"  Correlacao exibida      : SIM ({corr_val:.4f})")
print(f"  Grafico gerado          : SIM")
print(f"  Tabela gold existe      : SIM ({total_gold} registros)")
print()
print("TASK-004: DONE")

display(df_final.select("data", "ipca", "boi_gordo", "variacao_ipca", "variacao_boi"))